# 11_Hybrid_Forecasting

Implements the revised HSEP forecasting architecture using Holt's Trend Forecasting, Exponential Smoothing, and Feature-Based Projection.

## Forecasting Strategy

Prophet was initially evaluated for long-term skill demand forecasting. However, due to limited annual observations per skill and dependency issues with CmdStan, the forecasting layer was redesigned.

The final forecasting strategy is:

- ≥10 years → Holt's Trend Forecasting
- 7–9 years → Exponential Smoothing
- &lt;7 years → Feature-Based Projection


In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from statsmodels.tsa.holtwinters import Holt, ExponentialSmoothing
import warnings
warnings.filterwarnings('ignore')

## Load Data

In [2]:
history = pd.read_csv(r'C:\Users\Saanvi\Downloads\Mentorship\Generated Datasets\skill_demand_history_clean.csv')
skills = pd.read_csv(r'C:\Users\Saanvi\Downloads\Mentorship\Generated Datasets\expanded_skill_master.csv')

history['skill'] = history['skill'].str.lower().str.strip()
skills['skill'] = skills['skill'].str.lower().str.strip()

print('History shape:', history.shape)
print('Skills shape:', skills.shape)
print('History years:', history['year'].min(), '–', history['year'].max())

History shape: (2304, 4)
Skills shape: (114, 9)
History years: 2013 – 2025


## Historical Coverage

In [3]:
coverage = (
    history.groupby('skill')['year']
    .nunique()
    .reset_index(name='history_length')
)

skills = skills.merge(coverage, on='skill', how='left')
skills['history_length'] = skills['history_length'].fillna(0).astype(int)

print(skills['history_length'].describe())

count    114.000000
mean      10.605263
std        3.664270
min        0.000000
25%       10.000000
50%       12.000000
75%       13.000000
max       13.000000
Name: history_length, dtype: float64


## Forecasting Functions

In [4]:
BASE_YEAR = history['year'].max()
HORIZONS = [1, 2, 3]

def holt_forecast(skill_name):
    df = history[history['skill'] == skill_name].sort_values('year')
    y = df['demand'].astype(float).values

    model = Holt(
        y,
        damped_trend=True,
        initialization_method="estimated"
    )

    fit = model.fit(optimized=True)

    preds = fit.forecast(len(HORIZONS))
    preds = np.clip(preds, 0, None)

    return preds[0], preds[1], preds[2]

def exp_smoothing_forecast(skill_name):
    df = history[history['skill'] == skill_name].sort_values('year')
    y = df['demand'].astype(float).values

    model = ExponentialSmoothing(
        y,
        trend='add',
        seasonal=None
    )

    fit = model.fit(optimized=True)

    preds = fit.forecast(len(HORIZONS))
    preds = np.clip(preds, 0, None)

    return preds[0], preds[1], preds[2]

## Feature-Based Projection

In [5]:
FEATURE_COLS = [
    'linkedin_demand',
    'future_interest',
    'growth_rate',
    'current_usage',
    'global_adoption_score'
]

WEIGHTS = [0.30, 0.25, 0.25, 0.10, 0.10]

scaler_feat = MinMaxScaler()

feat_matrix = skills[FEATURE_COLS].fillna(0).values
feat_norm = scaler_feat.fit_transform(feat_matrix)

feat_df = pd.DataFrame(
    feat_norm,
    columns=FEATURE_COLS,
    index=skills.index
)

skills_norm = skills.copy()
skills_norm[FEATURE_COLS] = feat_df

latest_demand = (
    history[history['year'] == BASE_YEAR]
    .set_index('skill')['demand']
)

def feature_based_forecast(row):
    sk = row['skill']

    if sk in latest_demand.index:
        base = float(latest_demand[sk])
    else:
        base = float(latest_demand.median())

    growth_idx = sum(
        w * row[f]
        for w, f in zip(WEIGHTS, FEATURE_COLS)
        if not pd.isna(row[f])
    )

    growth_idx = min(growth_idx, 1.0)

    annual_growth = growth_idx * 0.15

    preds = [
        base * ((1 + annual_growth) ** h)
        for h in HORIZONS
    ]

    return preds[0], preds[1], preds[2]

## Main Forecast Loop

In [6]:
results = []

for _, row in skills_norm.iterrows():

    skill = row['skill']
    n_yrs = int(row['history_length'])

    try:
        if n_yrs >= 10:
            p1, p2, p3 = holt_forecast(skill)
            method = 'Holt'
            confidence = 0.85

        elif n_yrs >= 7:
            p1, p2, p3 = exp_smoothing_forecast(skill)
            method = 'ExponentialSmoothing'
            confidence = 0.65

        else:
            p1, p2, p3 = feature_based_forecast(row)
            method = 'FeatureBased'
            confidence = 0.35 + 0.05 * n_yrs

    except Exception as e:
        p1, p2, p3 = feature_based_forecast(row)
        method = 'FeatureBased'
        confidence = 0.30

    results.append({
        'skill': skill,
        'history_length': n_yrs,
        'forecast_method': method,
        'confidence': round(confidence, 2),
        'forecast_1y': round(p1, 4),
        'forecast_2y': round(p2, 4),
        'forecast_3y': round(p3, 4),
    })

forecast_df = pd.DataFrame(results)

print('Forecast shape:', forecast_df.shape)
print('Method distribution:')
print(forecast_df['forecast_method'].value_counts())

Forecast shape: (114, 7)
Method distribution:
forecast_method
Holt                    88
ExponentialSmoothing    16
FeatureBased            10
Name: count, dtype: int64


## Forecast Intelligence Scores

In [7]:
sc = MinMaxScaler()

forecast_df['forecast_score_1y'] = sc.fit_transform(forecast_df[['forecast_1y']])
forecast_df['forecast_score_2y'] = sc.fit_transform(forecast_df[['forecast_2y']])
forecast_df['forecast_score_3y'] = sc.fit_transform(forecast_df[['forecast_3y']])

forecast_df['forecast_score'] = (
    0.25 * forecast_df['forecast_score_1y'] +
    0.35 * forecast_df['forecast_score_2y'] +
    0.40 * forecast_df['forecast_score_3y']
)

def trend_label(row):
    change = (
        row['forecast_3y'] - row['forecast_1y']
    ) / max(abs(row['forecast_1y']), 1)

    if change > 0.30:
        return 'Explosive'
    if change > 0.10:
        return 'Growing'
    if change > 0.00:
        return 'Stable'
    return 'Declining'

forecast_df['forecast_trend'] = forecast_df.apply(trend_label, axis=1)

print('Trend distribution:')
print(forecast_df['forecast_trend'].value_counts())

Trend distribution:
forecast_trend
Stable       63
Growing      46
Declining     4
Explosive     1
Name: count, dtype: int64


## Save Output

In [8]:
cols = [
    'skill',
    'history_length',
    'forecast_method',
    'confidence',
    'forecast_1y',
    'forecast_2y',
    'forecast_3y',
    'forecast_score_1y',
    'forecast_score_2y',
    'forecast_score_3y',
    'forecast_score',
    'forecast_trend'
]

forecast_df[cols].to_csv(r'C:\Users\Saanvi\Downloads\Mentorship\Generated Datasets\forecast_results.csv',
    index=False
)

print('Saved : forecast_results.csv')
forecast_df[cols].head(10)

Saved : forecast_results.csv


,skill,history_length,forecast_method,confidence,forecast_1y,forecast_2y,forecast_3y,forecast_score_1y,forecast_score_2y,forecast_score_3y,forecast_score,forecast_trend
0,data analysis,13,Holt,0.85,17373.4729,17803.1507,18188.7957,0.061127,0.060784,0.059388,0.060311,Stable
1,excel,13,Holt,0.85,12667.4730,13033.4330,13370.5747,0.044255,0.044499,0.043656,0.044101,Stable
2,quality assurance,13,Holt,0.85,31809.6707,33259.7357,34669.4483,0.112884,0.113556,0.113199,0.113245,Stable
3,microsoft excel,13,Holt,0.85,11280.0779,11817.5939,12343.5015,0.039281,0.040348,0.040303,0.040063,Stable
4,python,13,Holt,0.85,217685.3807,220769.2226,223311.0592,0.779286,0.753752,0.729129,0.750286,Stable
5,sql,13,Holt,0.85,208242.1689,216057.9592,223507.8726,0.745430,0.737667,0.729772,0.736449,Stable
6,electrical engineering,13,Holt,0.85,13859.1489,14330.7235,14773.4543,0.048527,0.048928,0.048237,0.048551,Stable
7,mechanical engineering,13,Holt,0.85,21374.1985,22399.1535,23402.0670,0.075470,0.076475,0.076410,0.076198,Stable
8,java,13,Holt,0.85,134525.8347,132002.9278,129748.6070,0.481142,0.450685,0.423640,0.447481,Declining
9,aws,13,Holt,0.85,203971.4463,210834.5529,217203.7061,0.730119,0.719833,0.709188,0.718146,Stable


In [9]:
history[
    history["skill"]=="python"
].sort_values("year")

,skill,year,demand,adoption_rate
1765,python,2013,20873,0.033649
1766,python,2014,60220,0.054422
1767,python,2015,92132,0.058195
1768,python,2016,110838,0.049929
1769,python,2017,142894,0.051243
1770,python,2018,142575,0.043715
1771,python,2019,164245,0.044561
1772,python,2020,178461,0.044481
1773,python,2021,188103,0.042737
1774,python,2022,193947,0.041152
